In [51]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW

In [52]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("./prepared_balanced_nilm")

APPLIANCE = "boiler"
TARGET_BUILDING = "building_04"

TRAIN_BUILDINGS = ["building_01", "building_02"]
VAL_BUILDING = "building_03"

WINDOW_SIZE = 129
BATCH_SIZE = 256
EVAL_BATCH_SIZE = 2048
NUM_WORKERS = 0

CLASSIFIER_EPOCHS = 6
REGRESSOR_EPOCHS = 6
CALIBRATION_EPOCHS = 1

CLASSIFIER_LR = 1e-4
REGRESSOR_LR = 3e-5

POWER_THRESHOLDS = (5, 10, 15, 20, 25, 30, 40, 50)
ACTIVITY_THRESHOLD = 2.0
CALIBRATION_RATIO = 0.05
CALIBRATION_MODE = "all"
INACTIVE_KEEP_RATIO = 0.10
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [53]:
def building_dir(building):
    return DATA_ROOT / building

def csv_path(building):
    return building_dir(building) / "prepared_timeseries.csv"

def metadata_path(building):
    return building_dir(building) / "metadata.json"

def center_path(building, appliance, kind):
    return building_dir(building) / f"{appliance}_{kind}_centers.npy"

AGGREGATE_COLUMN = "aggregate_norm"

def target_norm_column(appliance):
    return f"{appliance}_norm"

def target_weight_column(appliance):
    return f"{appliance}_weight"

def load_building_dataframe(building):
    return pd.read_csv(csv_path(building))

def load_building_metadata(building):
    with open(metadata_path(building), "r", encoding="utf-8") as f:
        return json.load(f)

def get_appliance_metadata(building, appliance):
    meta = load_building_metadata(building)
    if appliance in meta and isinstance(meta[appliance], dict):
        return meta[appliance]
    return meta

def infer_columns(df, appliance):
    agg_col = AGGREGATE_COLUMN
    tgt_col = target_norm_column(appliance)
    if agg_col not in df.columns:
        raise ValueError(f"Missing aggregate column: {agg_col}")
    if tgt_col not in df.columns:
        raise ValueError(f"Missing target column: {tgt_col}")
    return agg_col, tgt_col

def inverse_target_transform_torch(y_norm, meta):
    mean_ = float(meta.get("target_mean", meta.get("mean", 0.0)))
    std_ = float(meta.get("target_std", meta.get("std", 1.0)))
    use_log = bool(meta.get("target_log1p", meta.get("use_log", False)))
    y_t = y_norm * std_ + mean_
    if use_log:
        y = torch.expm1(y_t)
    else:
        y = y_t
    return torch.clamp(y, min=0.0)

In [54]:
class RegressionDataset(Dataset):
    def __init__(self, aggregate, target, centers, window_size=129, sample_weights=None):
        self.aggregate = aggregate.astype(np.float32)
        self.target = target.astype(np.float32)
        self.centers = centers.astype(np.int64)
        self.window_size = window_size
        self.half = window_size // 2

        if sample_weights is None:
            self.sample_weights = np.ones(len(self.target), dtype=np.float32)
        else:
            self.sample_weights = sample_weights.astype(np.float32)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        c = int(self.centers[idx])
        l = c - self.half
        r = c + self.half + 1

        x = self.aggregate[l:r]
        y = self.target[c]
        w = self.sample_weights[c]

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        w = torch.tensor(w, dtype=torch.float32)
        return x, y, w

In [55]:
# df = load_building_dataframe(TARGET_BUILDING)
# print(df.columns.tolist())
# print([c for c in df.columns if c.endswith("_norm")])

In [56]:
def make_sample_weights(target_array, power_weight=1.0):
    t = np.abs(target_array.astype(np.float32))
    return 1.0 + power_weight * t

def prepare_building_arrays(building, appliance):
    df = load_building_dataframe(building)
    agg_col, tgt_col = infer_columns(df, appliance)

    aggregate = df[agg_col].to_numpy(dtype=np.float32)
    target = df[tgt_col].to_numpy(dtype=np.float32)

    w_col = target_weight_column(appliance)
    if w_col in df.columns:
        sample_weights = df[w_col].to_numpy(dtype=np.float32)
    else:
        sample_weights = make_sample_weights(target)

    all_centers = np.load(center_path(building, appliance, "all")).astype(np.int64)

    def load_optional(kind):
        p = center_path(building, appliance, kind)
        return np.load(p).astype(np.int64) if p.exists() else np.array([], dtype=np.int64)

    active_centers = load_optional("active")
    inactive_centers = load_optional("inactive")
    event_centers = load_optional("event")
    balanced_centers = load_optional("balanced")

    return {
        "aggregate": aggregate,
        "target": target,
        "weights": sample_weights,
        "all_centers": all_centers,
        "active_centers": active_centers,
        "inactive_centers": inactive_centers,
        "event_centers": event_centers,
        "balanced_centers": balanced_centers
    }

In [57]:
def sample_training_centers(data, inactive_keep_ratio=0.10):
    active = data["active_centers"]
    event = data["event_centers"]
    inactive = data["inactive_centers"]

    chosen = []

    if len(active) > 0:
        chosen.append(active)

    if len(event) > 0:
        chosen.append(event)

    if len(inactive) > 0 and inactive_keep_ratio > 0:
        n_inactive = max(1, int(len(inactive) * inactive_keep_ratio))
        rng = np.random.default_rng(SEED)
        inactive_sel = np.sort(rng.choice(inactive, size=min(n_inactive, len(inactive)), replace=False))
        chosen.append(inactive_sel)

    if len(chosen) == 0:
        return data["all_centers"]

    return np.sort(np.unique(np.concatenate(chosen)))

In [58]:
def make_target_regression_loaders(appliance, target_building, calibration_ratio=0.05, batch_size=256, window_size=129):
    data = prepare_building_arrays(target_building, appliance)

    aggregate = data["aggregate"]
    target = data["target"]
    weights = data["weights"]
    all_centers = data["all_centers"]

    rng = np.random.default_rng(SEED)
    pool = all_centers.copy()
    rng.shuffle(pool)

    n_cal = max(1, int(len(pool) * calibration_ratio))
    cal_centers = np.sort(pool[:n_cal])

    cal_set = set(cal_centers.tolist())
    holdout_centers = np.array([c for c in all_centers if int(c) not in cal_set], dtype=np.int64)

    if len(holdout_centers) == 0 and len(all_centers) > 0:
        holdout_centers = all_centers[:1]

    cal_ds = RegressionDataset(
        aggregate, target, cal_centers,
        window_size=window_size, sample_weights=weights
    )

    full_ds = RegressionDataset(
        aggregate, target, holdout_centers,
        window_size=window_size, sample_weights=weights
    )

    cal_loader = DataLoader(
        cal_ds, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    full_loader = DataLoader(
        full_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

    return cal_loader, full_loader

def build_regression_loaders(appliance, target_building, calibration_ratio=0.05):
    train_sets = []

    for b in TRAIN_BUILDINGS:
        data = prepare_building_arrays(b, appliance)

        train_centers = sample_training_centers(
            data,
            inactive_keep_ratio=INACTIVE_KEEP_RATIO
        )

        ds = RegressionDataset(
            data["aggregate"],
            data["target"],
            train_centers,
            window_size=WINDOW_SIZE,
            sample_weights=data["weights"]
        )
        train_sets.append(ds)

    train_ds = ConcatDataset(train_sets)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    calibration_loader, full_holdout_loader = make_target_regression_loaders(
        appliance=appliance,
        target_building=target_building,
        calibration_ratio=calibration_ratio,
        batch_size=BATCH_SIZE,
        window_size=WINDOW_SIZE
    )

    return train_loader, calibration_loader, full_holdout_loader

In [59]:
class PowerRegressor(nn.Module):
    def __init__(self, input_dim=1, channels=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 32, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, channels, 5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        return self.net(x).squeeze(-1)

In [60]:
class WeightedAsymmetricHuberLoss(nn.Module):
    def __init__(self, delta=1.0, underpredict_weight=2.0):
        super().__init__()
        self.delta = delta
        self.underpredict_weight = underpredict_weight

    def forward(self, pred, target, sample_weight=None):
        err = pred - target
        abs_err = torch.abs(err)

        huber = torch.where(
            abs_err <= self.delta,
            0.5 * err ** 2,
            self.delta * (abs_err - 0.5 * self.delta)
        )

        asym = torch.where(pred < target, self.underpredict_weight, 1.0)
        loss = huber * asym

        if sample_weight is not None:
            loss = loss * sample_weight

        return loss.mean()

In [61]:
def train_one_epoch_regressor(model, loader, optimizer, loss_fn, device):
    model.train()
    losses = []

    for x, y, w in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        w = w.to(device, non_blocking=True)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y, w)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses)) if len(losses) else 0.0

In [62]:
def collect_outputs_regression_only(regressor, loader, device, meta):
    regressor.eval()

    preds, trues = [], []

    with torch.inference_mode():
        for x, y, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            reg = regressor(x)

            reg_np = inverse_target_transform_torch(reg, meta).cpu().numpy()
            true_np = inverse_target_transform_torch(y, meta).cpu().numpy()

            preds.append(reg_np)
            trues.append(true_np)

    if len(preds) == 0:
        return None, None

    return np.concatenate(preds), np.concatenate(trues)

In [63]:
def regression_metrics(y_true, y_pred):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot
    rel_mae_pct = float(100.0 * mae / max(np.mean(y_true), 1e-8))

    return {
        "mae": mae,
        "r2": r2,
        "rel_mae_pct": rel_mae_pct,
        "pred_mean": float(np.mean(y_pred)),
        "true_mean": float(np.mean(y_true)),
        "num_samples": len(y_true)
    }

def binary_metrics_from_power(y_true, y_pred_power, pred_power_threshold=20.0, activity_threshold=20.0):
    true_on = (y_true >= activity_threshold).astype(np.int32)
    pred_on = (y_pred_power >= pred_power_threshold).astype(np.int32)

    tp = int(np.sum((true_on == 1) & (pred_on == 1)))
    tn = int(np.sum((true_on == 0) & (pred_on == 0)))
    fp = int(np.sum((true_on == 0) & (pred_on == 1)))
    fn = int(np.sum((true_on == 1) & (pred_on == 0)))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": acc
    }

In [64]:
def sweep_power_thresholds(regressor, loader, device, meta,
                           thresholds=(0.5, 1, 2, 3, 5, 7, 10, 15, 20),
                           activity_threshold=20.0):
    preds, trues = collect_outputs_regression_only(regressor, loader, device, meta)

    rows = []
    for thr in thresholds:
        reg = regression_metrics(trues, preds)
        binm = binary_metrics_from_power(
            trues, preds,
            pred_power_threshold=thr,
            activity_threshold=activity_threshold
        )

        rows.append({
            "pred_power_threshold": thr,
            "mae": reg["mae"],
            "r2": reg["r2"],
            "rel_mae_pct": reg["rel_mae_pct"],
            "pred_mean": reg["pred_mean"],
            "true_mean": reg["true_mean"],
            "precision": binm["precision"],
            "recall": binm["recall"],
            "f1": binm["f1"],
            "accuracy": binm["accuracy"]
        })

    return pd.DataFrame(rows)

In [65]:
def choose_best_power_threshold(sweep_df, min_recall=0.05):
    df = sweep_df.copy()
    usable = df[df["recall"] >= min_recall].copy()

    if len(usable) == 0:
        usable = df.copy()

    usable = usable.sort_values(
        by=["f1", "precision", "mae"],
        ascending=[False, False, True]
    )
    return float(usable.iloc[0]["pred_power_threshold"])

In [66]:
train_loader, calibration_loader, full_holdout_loader = build_regression_loaders(
    APPLIANCE, TARGET_BUILDING, calibration_ratio=CALIBRATION_RATIO
)

target_meta = get_appliance_metadata(TARGET_BUILDING, APPLIANCE)

regressor = PowerRegressor().to(DEVICE)
reg_loss_fn = WeightedAsymmetricHuberLoss(delta=1.0, underpredict_weight=3.0)
opt_reg = AdamW(regressor.parameters(), lr=REGRESSOR_LR, weight_decay=1e-4)

In [67]:
for epoch in range(1, REGRESSOR_EPOCHS + 1):
    t0 = time.time()
    reg_loss = train_one_epoch_regressor(regressor, train_loader, opt_reg, reg_loss_fn, DEVICE)
    print(f"[REG {epoch:02d}/{REGRESSOR_EPOCHS}] loss={reg_loss:.4f} time={time.time()-t0:.1f}s")

[REG 01/6] loss=29.3213 time=2.4s
[REG 02/6] loss=17.7739 time=1.9s
[REG 03/6] loss=2.6824 time=1.8s
[REG 04/6] loss=2.3145 time=1.8s
[REG 05/6] loss=2.2018 time=2.2s
[REG 06/6] loss=2.1081 time=2.3s


In [68]:
full_preds, full_trues = collect_outputs_regression_only(
    regressor, full_holdout_loader, DEVICE, target_meta
)

raw_reg = regression_metrics(full_trues, full_preds)

print("RAW REGRESSION")
for k, v in raw_reg.items():
    print(f"{k}: {v}")

RAW REGRESSION
mae: 1.6700749397277832
r2: -5.636588743442578
rel_mae_pct: 399.7060546875
pred_mean: 2.0509073734283447
true_mean: 0.4178257882595062
num_samples: 15409411


In [69]:
power_sweep_df = sweep_power_thresholds(
    regressor=regressor,
    loader=full_holdout_loader,
    device=DEVICE,
    meta=target_meta,
    thresholds=POWER_THRESHOLDS,
    activity_threshold=ACTIVITY_THRESHOLD
)

print(power_sweep_df.sort_values(["f1", "mae"], ascending=[False, True]))

   pred_power_threshold       mae        r2  rel_mae_pct  pred_mean  \
0                     5  1.670075 -5.636589   399.706055   2.050907   
1                    10  1.670075 -5.636589   399.706055   2.050907   
2                    15  1.670075 -5.636589   399.706055   2.050907   
3                    20  1.670075 -5.636589   399.706055   2.050907   
4                    25  1.670075 -5.636589   399.706055   2.050907   
5                    30  1.670075 -5.636589   399.706055   2.050907   
6                    40  1.670075 -5.636589   399.706055   2.050907   
7                    50  1.670075 -5.636589   399.706055   2.050907   

   true_mean  precision  recall   f1  accuracy  
0   0.417826        0.0     0.0  0.0  0.950938  
1   0.417826        0.0     0.0  0.0  0.950938  
2   0.417826        0.0     0.0  0.0  0.950938  
3   0.417826        0.0     0.0  0.0  0.950938  
4   0.417826        0.0     0.0  0.0  0.950938  
5   0.417826        0.0     0.0  0.0  0.950938  
6   0.417826     

In [70]:
BEST_POWER_THRESHOLD = choose_best_power_threshold(power_sweep_df, min_recall=0.05)
print("Selected predicted-power threshold:", BEST_POWER_THRESHOLD)

Selected predicted-power threshold: 5.0


In [71]:
final_bin = binary_metrics_from_power(
    full_trues,
    full_preds,
    pred_power_threshold=BEST_POWER_THRESHOLD,
    activity_threshold=ACTIVITY_THRESHOLD
)

print("\nBINARY")
for k, v in final_bin.items():
    print(f"{k}: {v}")


BINARY
tp: 0
tn: 14653397
fp: 0
fn: 756014
precision: 0.0
recall: 0.0
f1: 0.0
accuracy: 0.9509381636974963


In [72]:
print("Pred stats")
print("min:", float(np.min(full_preds)))
print("p1 :", float(np.percentile(full_preds, 1)))
print("p10:", float(np.percentile(full_preds, 10)))
print("p50:", float(np.percentile(full_preds, 50)))
print("p90:", float(np.percentile(full_preds, 90)))
print("p99:", float(np.percentile(full_preds, 99)))
print("max:", float(np.max(full_preds)))

print("\nTrue stats")
print("min:", float(np.min(full_trues)))
print("p1 :", float(np.percentile(full_trues, 1)))
print("p10:", float(np.percentile(full_trues, 10)))
print("p50:", float(np.percentile(full_trues, 50)))
print("p90:", float(np.percentile(full_trues, 90)))
print("p99:", float(np.percentile(full_trues, 99)))
print("max:", float(np.max(full_trues)))

Pred stats
min: 1.573491096496582
p1 : 1.8144562244415283
p10: 1.91885244846344
p50: 2.048398971557617
p90: 2.1799840927124023
p99: 2.291020154953003
max: 4.385916709899902

True stats
min: 0.0
p1 : 0.0
p10: 0.0
p50: 0.0
p90: 1.5102742910385132
p99: 2.778642416000366
max: 3.0537030696868896


In [73]:
# full_preds, full_trues = collect_outputs_regression_only(
#     regressor, full_holdout_loader, DEVICE, target_meta
# )

# raw_reg = regression_metrics(full_trues, full_preds)
# corrected_preds = apply_linear_bias_correction(full_preds, correction)
# corr_reg = regression_metrics(full_trues, corrected_preds)

# print("RAW REGRESSION")
# for k, v in raw_reg.items():
#     print(f"{k}: {v}")

# print("\nCORRECTED REGRESSION")
# for k, v in corr_reg.items():
#     print(f"{k}: {v}")

In [74]:
# power_sweep_df = sweep_power_thresholds(
#     regressor=regressor,
#     loader=full_holdout_loader,
#     device=DEVICE,
#     meta=target_meta,
#     thresholds=POWER_THRESHOLDS,
#     activity_threshold=ACTIVITY_THRESHOLD,
#     correction=correction
# )

# print(power_sweep_df.sort_values(["f1", "mae"], ascending=[False, True]))

In [75]:
# BEST_POWER_THRESHOLD = choose_best_power_threshold(power_sweep_df, min_recall=0.10)
# print("Selected predicted-power threshold:", BEST_POWER_THRESHOLD)

In [76]:
# final_bin = binary_metrics_from_power(
#     full_trues,
#     corrected_preds,
#     pred_power_threshold=BEST_POWER_THRESHOLD,
#     activity_threshold=ACTIVITY_THRESHOLD
# )

# print("\nBINARY")
# for k, v in final_bin.items():
#     print(f"{k}: {v}")